<a href="https://colab.research.google.com/github/pgianinh50364/nlp_course/blob/2024/week02_classification/homework_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Salary prediction, episode II: make it actually work (4 points)

Your main task is to use some of the tricks you've learned on the network and analyze if you can improve __validation MAE__. Try __at least 3 options__ from the list below for a passing grade. Write a short report about what you have tried. More ideas = more bonus points.

__Please be serious:__ " plot learning curves in MAE/epoch, compare models based on optimal performance, test one change at a time. You know the drill :)

You can use either __pytorch__ or __tensorflow__ or any other framework (e.g. pure __keras__). Feel free to adapt the seminar code for your needs. For tensorflow version, consider `seminar_tf2.ipynb` as a starting point.


In [ ]:
!pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.3/927.3 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.3/819.3 kB 40.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.tokenize import TweetTokenizer
import gensim.downloader

import torch
from torch import nn
from torch.nn import functional as F
import lightning as L
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.rnn import pad_sequence

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import warnings
%matplotlib inline
warnings.filterwarnings('ignore')

import gc

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/pgianinh50364/nlp_course/refs/heads/2024/week02_classification/comments.tsv', sep='\t')

In [ ]:
data.head()

,should_ban,comment_text
0,0,The picture on the article is not of the actor...
1,1,"Its madness. Shes of Chinese heritage, but JAP..."
2,1,Fuck You. Why don't you suck a turd out of my ...
3,1,God is dead\nI don't mean to startle anyone bu...
4,1,THIS USER IS A PLANT FROM BRUCE PERENS AND GRO...


In [ ]:
print('Average word length of words in data is {0:.0f}.'.format(np.mean(data['comment_text'].apply(lambda x: len(x.split())))))
print('Max word length of words in data is {0:.0f}'.format(np.max(data['comment_text'].apply(lambda x: len(x.split())))))
print('Min word length of words in data is {0:.0f}'.format(np.min(data['comment_text'].apply(lambda x: len(x.split())))))

Average word length of words in data is 60.
Max word length of words in data is 1069
Min word length of words in data is 2


## Feature Engineering

In [ ]:
tk = TweetTokenizer()
data['comment_text'] = data['comment_text'].apply(lambda x: tk.tokenize(x))

In [ ]:
X_train, y_train, X_test, y_test = train_test_split(data['comment_text'], data['should_ban'], test_size=0.1, random_state=42)

In [ ]:
gensim.downloader.info()['models'].keys()

dict_keys(['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis'])

In [ ]:
embeddings = gensim.downloader.load('glove-twitter-200')

[==================================================] 100.0% 758.5/758.5MB downloaded


In [ ]:
class MyCNN(L.LightningModule):
  def __init__(self, input_dims, hidden_dims, drop_out, kernel_size):
    super().__init__()
    self.input_dims = input_dims
    self.hidden_dims = hidden_dims
    self.kernel_size = kernel_size
    self.drop_out = drop_out

    self.conv1d = nn.Conv1d(input_dims, hidden_dims, kernel_size=kernel_size)
    self.batchnorm = nn.BatchNorm1d(hidden_dims)
    self.dropout = nn.Dropout(drop_out)

  def forward(self, x):
    x = self.conv1d(x)
    x = self.batchnorm(x)
    x = self.dropout(x)
    return x

class MyBiLSTM(L.LightningModule):
  def __init__(self, input_dims, hidden_dims):
    super().__init__()
    self.input_dims = input_dims
    self.hidden_dims = hidden_dims

    self.lstm = nn.LSTM(input_dims, hidden_dims, bidirectional=True, batch_first=True)

  def forward(self, x):
    self.lstm.flatten_parameters()
    x, _ = self.lstm(x)
    return x.transpose(-1, -2)

class AttnPooling(L.LightningModule):
  def __init__(self, input_dims):
    super().__init__()
    self.input_dims = input_dims

    self.attn = nn.Linear(input_dims, 1)

  def forward(self, batch):
    self.attn = self.attn(batch)
    self.attn = F.softmax(self.attn, dim=1)
    return torch.sum(batch * self.attn, dim=1)

In [ ]:
  class CNNwEmbeddings(L.LightningModule):
    def __init__(self, embedding_dim, hidden_dims, num_classes, input_length, vocab_size, drop_out, learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.embedding = nn.Embedding.from_pretrained(torch.Tensor(embeddings))
        self.embedding.weight.requires_grad_()
        self.input_length = input_length
        self.drop_out = drop_out
        self.conv1 = nn.Conv1d(embedding_dim, hidden_dims, kernel_size=3)
        self.bn1 = nn.BatchNorm1d(hidden_dims)
        self.dropout1 = nn.Dropout(drop_out)
        self.conv2 = nn.Conv1d(hidden_dims, hidden_dims//2, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(hidden_dims//2)
        self.dropout2 = nn.Dropout(drop_out)
        self.pool = nn.MaxPool1d(kernel_size=2)
        self.fc = nn.Linear(hidden_dims//2 * ((input_length - 6) // 2 + 1), num_classes)
        self.learning_rate = learning_rate

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = self.bn1(x)
        x = self.dropout1(x)
        x = F.relu(self.conv2(x))
        x = self.bn2(x)
        x = self.dropout2(x)
        x = self.pool(x)
        x = x.flatten(start_dim=1)
        x = self.fc(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", self.train_acc(y_hat, y), prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("val_loss", loss,prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer




### A short report

Please tell us what you did and how did it work.

`<YOUR_TEXT_HERE>`, i guess...

## Recommended options

#### A) CNN architecture

All the tricks you know about dense and convolutional neural networks apply here as well.
* Dropout. Nuff said.
* Batch Norm. This time it's `nn.BatchNorm*`/`L.BatchNormalization`
* Parallel convolution layers. The idea is that you apply several nn.Conv1d to the same embeddings and concatenate output channels.
* More layers, more neurons, ya know...


#### B) Play with pooling

There's more than one way to perform pooling:
* Max over time (independently for each feature)
* Average over time (excluding PAD)
* Softmax-pooling:
$$ out_{i, t} = \sum_t {h_{i,t} \cdot {{e ^ {h_{i, t}}} \over \sum_\tau e ^ {h_{j, \tau}} } }$$

* Attentive pooling
$$ out_{i, t} = \sum_t {h_{i,t} \cdot Attn(h_t)}$$

, where $$ Attn(h_t) = {{e ^ {NN_{attn}(h_t)}} \over \sum_\tau e ^ {NN_{attn}(h_\tau)}}  $$
and $NN_{attn}$ is a dense layer.

The optimal score is usually achieved by concatenating several different poolings, including several attentive pooling with different $NN_{attn}$ (aka multi-headed attention).

The catch is that keras layers do not inlude those toys. You will have to [write your own keras layer](https://keras.io/layers/writing-your-own-keras-layers/). Or use pure tensorflow, it might even be easier :)

#### C) Fun with words

It's not always a good idea to train embeddings from scratch. Here's a few tricks:

* Use a pre-trained embeddings from `gensim.downloader.load`. See last lecture.
* Start with pre-trained embeddings, then fine-tune them with gradient descent. You may or may not download pre-trained embeddings from [here](http://nlp.stanford.edu/data/glove.6B.zip) and follow this [manual](https://keras.io/examples/nlp/pretrained_word_embeddings/) to initialize your Keras embedding layer with downloaded weights.
* Use the same embedding matrix in title and desc vectorizer


#### D) Going recurrent

We've already learned that recurrent networks can do cool stuff in sequence modelling. Turns out, they're not useless for classification as well. With some tricks of course..

* Like convolutional layers, LSTM should be pooled into a fixed-size vector with some of the poolings.
* Since you know all the text in advance, use bidirectional RNN
  * Run one LSTM from left to right
  * Run another in parallel from right to left
  * Concatenate their output sequences along unit axis (dim=-1)

* It might be good idea to mix convolutions and recurrent layers differently for title and description


#### E) Optimizing seriously

* You don't necessarily need 100 epochs. Use early stopping. If you've never done this before, take a look at [early stopping callback(keras)](https://keras.io/callbacks/#earlystopping) or in [pytorch(lightning)](https://pytorch-lightning.readthedocs.io/en/latest/common/early_stopping.html).
  * In short, train until you notice that validation
  * Maintain the best-on-validation snapshot via `model.save(file_name)`
  * Plotting learning curves is usually a good idea
  
Good luck! And may the force be with you!